In [1]:
from array import array
from collections import deque, defaultdict, Counter, namedtuple

# Student Identity
LAST_NAME = "PALAOAG"
STUDENT_ID = "TUPM-26-1949"
FAVORITE_ARTIST = "RICO BLANCO"
SEED_NUM = int(STUDENT_ID[-1])

# Identity-Based Computational Seed (immutable tuples)
LAST_NAME    = LAST_NAME.upper().replace(" ", "")
SEED_DIGITS = (SEED_NUM,)
ARTIST_CODES = tuple(ord(ch) for ch in FAVORITE_ARTIST.upper() if ch.isalpha())
IDENTITY_SEED = sum(SEED_DIGITS) * len(ARTIST_CODES) + sum(c * (i + 1) for i, c in enumerate(ARTIST_CODES))

def make_stream():
    """Fresh number generator. Two deques are rotated on every call, so the
    SEED_NUM digits and FAVORITE_ARTIST character positions keep cycling."""
    digit_ring = deque(SEED_DIGITS)
    code_ring = deque(ARTIST_CODES)
    state = {"step": 0}

    def next_value(lo, hi):
        step = state["step"]
        pos = step % len(code_ring) + 1        
        n = (digit_ring[0] + 1) * code_ring[0] * pos + code_ring[-1] + step * 7 + IDENTITY_SEED % 13
        digit_ring.rotate(-1)
        code_ring.rotate(-1)
        state["step"] += 1
        return lo + n % (hi - lo + 1)

    return next_value

def table(columns, rows):
    line = " | ".join(f"{title:<{w}}" for title, w in columns)
    print(line)
    print("-" * len(line))
    for r in rows:
        print(" | ".join(f"{str(v):<{w}}" for v, (_, w) in zip(r, columns)))

def mean(values):
    return sum(values) / len(values) if len(values) else 0.0

print("IDENTITY_SEED =", IDENTITY_SEED)
print("SEED_DIGITS   =", SEED_DIGITS)
print("ARTIST_CODES  =", ARTIST_CODES)

IDENTITY_SEED = 4093
SEED_DIGITS   = (9,)
ARTIST_CODES  = (82, 73, 67, 79, 66, 76, 65, 78, 67, 79)


In [8]:
# CELL 2

Record = namedtuple("Record", "source category value")        

sources    = tuple(f"{LAST_NAME}_T{i}" for i in range(1, 6))   # T1..T5
categories = ("TEMPERATURE", "VOLTAGE", "CURRENT", "PRESSURE", "SPEED")

nxt = make_stream()
def reading():
    return round(nxt(10, 500) + nxt(0, 99) / 100, 2)

records = [Record(s, categories[j % 5], reading())
           for i, s in enumerate(sources) for j in (i, i + 1)]

records.append(Record(sources[0], categories[0], reading()))   
records.extend([Record(sources[i], c, reading())               
                for i in (2, 3) for c in ("TEMPERATURE", "VOLTAGE")])

table([("No.", 4), ("Source", 16), ("Category", 12), ("Value", 8)],
      [(n, r.source, r.category, r.value) for n, r in enumerate(records, 1)])

No.  | Source           | Category     | Value   
-------------------------------------------------
1    | PALAOAG_T1       | TEMPERATURE  | 429.6   
2    | PALAOAG_T1       | VOLTAGE      | 154.59  
3    | PALAOAG_T2       | VOLTAGE      | 482.72  
4    | PALAOAG_T2       | CURRENT      | 270.65  
5    | PALAOAG_T3       | CURRENT      | 293.41  
6    | PALAOAG_T3       | PRESSURE     | 499.3   
7    | PALAOAG_T4       | PRESSURE     | 224.29  
8    | PALAOAG_T4       | SPEED        | 61.42   
9    | PALAOAG_T5       | SPEED        | 340.35  
10   | PALAOAG_T5       | TEMPERATURE  | 363.11  
11   | PALAOAG_T1       | TEMPERATURE  | 78.0    
12   | PALAOAG_T3       | TEMPERATURE  | 294.99  
13   | PALAOAG_T3       | VOLTAGE      | 131.12  
14   | PALAOAG_T4       | TEMPERATURE  | 410.05  
15   | PALAOAG_T4       | VOLTAGE      | 433.81  


In [3]:
# CELL 3

organized = {}
for r in records:
    organized.setdefault(r.source, {}).setdefault(r.category, []).append(r.value)   

for src, cats in organized.items():
    print(src)
    for cat, vals in cats.items():
        print(f"   {cat:<12}: {vals}")

PALAOAG_T1
   TEMPERATURE : [429.6, 78.0]
   VOLTAGE     : [154.59]
PALAOAG_T2
   VOLTAGE     : [482.72]
   CURRENT     : [270.65]
PALAOAG_T3
   CURRENT     : [293.41]
   PRESSURE    : [499.3]
   TEMPERATURE : [294.99]
   VOLTAGE     : [131.12]
PALAOAG_T4
   PRESSURE    : [224.29]
   SPEED       : [61.42]
   TEMPERATURE : [410.05]
   VOLTAGE     : [433.81]
PALAOAG_T5
   SPEED       : [340.35]
   TEMPERATURE : [363.11]


In [4]:
# CELL 4

pair_counts = Counter((r.source, r.category) for r in records)
repeated    = {k: n for k, n in pair_counts.items() if n > 1}

unique_sources    = list(dict.fromkeys(r.source for r in records))     
unique_categories = list(dict.fromkeys(r.category for r in records))
unique_pairs      = set(pair_counts)

print("Repeated entries:")
for (s, c), n in repeated.items():
    print(f"   {s} / {c} appears {n} times -> {organized[s][c]}")
print("Distinct sources   :", len(unique_sources))
print("Distinct categories:", len(unique_categories))
print("Distinct source/category pairs:", len(unique_pairs))

cats_by_source = {s: {r.category for r in records if r.source == s} for s in sources}
a, b = cats_by_source[sources[0]], cats_by_source[sources[2]]
print(f"\nCategories of {sources[0]}: {sorted(a)}")
print(f"Categories of {sources[2]}: {sorted(b)}")
print("Union                :", sorted(a | b))
print("Intersection         :", sorted(a & b))
print("Difference (A - B)   :", sorted(a - b))
print("Symmetric difference :", sorted(a ^ b))

not_measured = {s: sorted(set(categories) - cats_by_source[s]) for s in sources}
print("\nNot measured per source:")
for s, missing in not_measured.items():
    print(f"   {s}: {missing}")

Repeated entries:
   PALAOAG_T1 / TEMPERATURE appears 2 times -> [429.6, 78.0]
Distinct sources   : 5
Distinct categories: 5
Distinct source/category pairs: 14

Categories of PALAOAG_T1: ['TEMPERATURE', 'VOLTAGE']
Categories of PALAOAG_T3: ['CURRENT', 'PRESSURE', 'TEMPERATURE', 'VOLTAGE']
Union                : ['CURRENT', 'PRESSURE', 'TEMPERATURE', 'VOLTAGE']
Intersection         : ['TEMPERATURE', 'VOLTAGE']
Difference (A - B)   : []
Symmetric difference : ['CURRENT', 'PRESSURE']

Not measured per source:
   PALAOAG_T1: ['CURRENT', 'PRESSURE', 'SPEED']
   PALAOAG_T2: ['PRESSURE', 'SPEED', 'TEMPERATURE']
   PALAOAG_T3: ['SPEED']
   PALAOAG_T4: ['CURRENT']
   PALAOAG_T5: ['CURRENT', 'PRESSURE', 'VOLTAGE']


In [5]:
# CELL 5

by_category = {c: array("d", [r.value for r in records if r.category == c]) for c in categories}
by_source   = {s: array("d", [r.value for r in records if r.source == s]) for s in sources}

def stats(arr):
    if not len(arr):                      
        return (0, 0, 0, 0, 0)
    return (len(arr), min(arr), max(arr), round(mean(arr), 2), round(sum(arr), 2))

print("Per category")
table([("Category", 12), ("Count", 5), ("Min", 8), ("Max", 8), ("Average", 8), ("Total", 9)],
      [(c, *stats(a)) for c, a in by_category.items()])
print("\nPer source")
table([("Source", 16), ("Count", 5), ("Min", 8), ("Max", 8), ("Average", 8), ("Total", 9)],
      [(s, *stats(a)) for s, a in by_source.items()])
print("\nUnknown category lookup (fault tolerant):", stats(by_category.get("HUMIDITY", array("d"))))

Per category
Category     | Count | Min      | Max      | Average  | Total    
-----------------------------------------------------------------
TEMPERATURE  | 5     | 78.0     | 429.6    | 315.15   | 1575.75  
VOLTAGE      | 4     | 131.12   | 482.72   | 300.56   | 1202.24  
CURRENT      | 2     | 270.65   | 293.41   | 282.03   | 564.06   
PRESSURE     | 2     | 224.29   | 499.3    | 361.8    | 723.59   
SPEED        | 2     | 61.42    | 340.35   | 200.89   | 401.77   

Per source
Source           | Count | Min      | Max      | Average  | Total    
---------------------------------------------------------------------
PALAOAG_T1       | 3     | 78.0     | 429.6    | 220.73   | 662.19   
PALAOAG_T2       | 2     | 270.65   | 482.72   | 376.69   | 753.37   
PALAOAG_T3       | 4     | 131.12   | 499.3    | 304.71   | 1218.82  
PALAOAG_T4       | 4     | 61.42    | 433.81   | 282.39   | 1129.57  
PALAOAG_T5       | 2     | 340.35   | 363.11   | 351.73   | 703.46   

Unknown category looku

In [6]:
# CELL 6

processed = {k: round(mean(organized[k[0]][k[1]]), 2) for k in pair_counts}
table([("Source", 16), ("Category", 12), ("Average", 8)],
      [(s, c, avg) for (s, c), avg in processed.items()])

ranking = deque(sorted(processed.items(), key=lambda kv: kv[1], reverse=True))
ordered_rows, rank = [], 1
while ranking:                                   
    (s, c), avg = ranking.popleft()
    ordered_rows.append((rank, s, c, avg))
    rank += 1
table([("Rank", 4), ("Source", 16), ("Category", 12), ("Average", 8)], ordered_rows)

recent = deque(records, maxlen=5)                
print("\nLast 5 records collected:")
for r in recent:
    print("  ", r)

Source           | Category     | Average 
------------------------------------------
PALAOAG_T1       | TEMPERATURE  | 253.8   
PALAOAG_T1       | VOLTAGE      | 154.59  
PALAOAG_T2       | VOLTAGE      | 482.72  
PALAOAG_T2       | CURRENT      | 270.65  
PALAOAG_T3       | CURRENT      | 293.41  
PALAOAG_T3       | PRESSURE     | 499.3   
PALAOAG_T4       | PRESSURE     | 224.29  
PALAOAG_T4       | SPEED        | 61.42   
PALAOAG_T5       | SPEED        | 340.35  
PALAOAG_T5       | TEMPERATURE  | 363.11  
PALAOAG_T3       | TEMPERATURE  | 294.99  
PALAOAG_T3       | VOLTAGE      | 131.12  
PALAOAG_T4       | TEMPERATURE  | 410.05  
PALAOAG_T4       | VOLTAGE      | 433.81  
Rank | Source           | Category     | Average 
-------------------------------------------------
1    | PALAOAG_T3       | PRESSURE     | 499.3   
2    | PALAOAG_T2       | VOLTAGE      | 482.72  
3    | PALAOAG_T4       | VOLTAGE      | 433.81  
4    | PALAOAG_T4       | TEMPERATURE  | 410.05  
5    | PALAO

In [7]:
# CELL 7

all_values = array("d", [r.value for r in records])
best_cat = max(by_category, key=lambda c: mean(by_category[c]))
best_src = max(by_source, key=lambda s: mean(by_source[s]))

print(f"Total records            : {len(records)}")
print(f"Distinct sources         : {len(unique_sources)}")
print(f"Distinct categories      : {len(unique_categories)}")
print(f"Repeated pairs           : {len(repeated)}")
print(f"Overall min / max        : {min(all_values)} / {max(all_values)}")
print(f"Overall average          : {mean(all_values):.2f}")
print(f"Highest-average category : {best_cat} ({mean(by_category[best_cat]):.2f})")
print(f"Highest-average source   : {best_src} ({mean(by_source[best_src]):.2f})")
print(f"Top processed result     : {ordered_rows[0][1]} / {ordered_rows[0][2]} = {ordered_rows[0][3]}")

Total records            : 15
Distinct sources         : 5
Distinct categories      : 5
Repeated pairs           : 1
Overall min / max        : 61.42 / 499.3
Overall average          : 297.83
Highest-average category : PRESSURE (361.80)
Highest-average source   : PALAOAG_T2 (376.69)
Top processed result     : PALAOAG_T3 / PRESSURE = 499.3
